<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_03_feature_engineering/stage_03b_feature_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03b_feature_application**

## **Introducción**

Esta notebook corresponde al stage_03 - Technical Indicators del pipeline neural_profit y tiene como objetivo generar, evaluar y consolidar indicadores técnicos intradía para el índice MNQ, a partir de datos minuto a minuto.

Partiendo del dataset intradía ya etiquetado con objetivos de retorno, se calculan indicadores técnicos de forma independiente por jornada, evitando la mezcla de información entre días. Esto garantiza consistencia temporal y previene leakage en etapas posteriores de modelado.

El proceso incluye la evaluación cuantitativa de los indicadores mediante Information Coefficient (IC), utilizando correlación de Spearman entre cada indicador y los targets de retorno definidos para distintos horizontes. Este análisis permite medir no solo la relación promedio con el target, sino también su estabilidad a lo largo del tiempo.

Como resultado final, se generan datasets consolidados y listos para modelado, que incluyen:

- Variables OHLCV
- Targets de retorno a distintos horizontes
- Indicadores técnicos seleccionados y validados

Estos artefactos serán utilizados en las siguientes etapas del pipeline para selección de features, entrenamiento y evaluación de modelos predictivos.

## **0. Configuración del Entorno**


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación e importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos

import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal



from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

In [3]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


In [4]:
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

### 0.3. Definición de rutas



In [5]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [6]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/processed/mnq_intraday.parquet"))
#IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/targets/mnq_intraday_targets.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
#OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

In [7]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
#IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
#OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

### 0.4. Códigos auxiliares para carga de datos y visualización


In [8]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [9]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")

# **1. Carga de dataset**

In [10]:
mnq_intraday = load_mnq_parquet()
info = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_intraday
Shape: (894845, 17)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 04:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 270, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 09:30:00+00:00  ->  2025-06-13 20:00:00+00:00


### Validación temporal de dataset `mnq_intraday`


In [11]:
import pandas as pd
import numpy as np

# ============================================================
# Validación temporal de mnq_intraday
# Ejecutar inmediatamente después de:
# mnq_intraday = load_mnq_parquet()
# ============================================================

def validate_mnq_intraday(df: pd.DataFrame, verbose: bool = True) -> dict:
    """
    Valida consistencia temporal básica para un dataset intradía.

    Chequeos:
    1) Índice datetime válido
    2) Orden cronológico global
    3) Duplicados de timestamp
    4) Consistencia de columna `date`
    5) Monotonía de `minute_of_day` dentro de cada día
    6) Duplicados de `minute_of_day` dentro de cada día
    7) Saltos temporales negativos o nulos
    8) Gaps intradía distintos de 1 minuto
    """

    result = {
        "ok": True,
        "checks": {},
        "summary": {},
        "artifacts": {}
    }

    df = df.copy()

    # ------------------------------------------------------------
    # 0) Verificaciones básicas de estructura
    # ------------------------------------------------------------
    required_cols = ["date", "minute_of_day", "open", "high", "low", "close", "volume"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice del DataFrame debe ser un pd.DatetimeIndex")

    if df.empty:
        raise ValueError("El DataFrame está vacío")

    # ------------------------------------------------------------
    # 1) Orden global del índice
    # ------------------------------------------------------------
    is_monotonic = df.index.is_monotonic_increasing
    has_unique_index = df.index.is_unique
    duplicated_index = df.index[df.index.duplicated()].unique()

    result["checks"]["index_is_monotonic_increasing"] = bool(is_monotonic)
    result["checks"]["index_is_unique"] = bool(has_unique_index)
    result["summary"]["n_duplicated_timestamps"] = int(len(duplicated_index))
    result["artifacts"]["duplicated_timestamps"] = duplicated_index

    # Si no está ordenado, mostramos evidencia pero no reordenamos silenciosamente
    if not is_monotonic:
        diffs_ns = pd.Series(df.index.view("i8")).diff()
        bad_order_pos = np.where(diffs_ns <= 0)[0]
        result["artifacts"]["bad_global_order_positions"] = bad_order_pos[:20]
        result["ok"] = False

    if not has_unique_index:
        result["ok"] = False

    # ------------------------------------------------------------
    # 2) Consistencia entre index.date y columna `date`
    # ------------------------------------------------------------
    # Normalizamos ambos a fecha sin hora
    index_dates = pd.Index(df.index.tz_localize(None).date if df.index.tz is not None else df.index.date)
    col_dates = pd.to_datetime(df["date"]).dt.date

    date_match = (index_dates == col_dates).all()
    result["checks"]["date_column_matches_index_date"] = bool(date_match)

    if not date_match:
        mismatch_mask = index_dates != col_dates
        mismatches = df.loc[mismatch_mask, ["date", "minute_of_day", "close"]].head(20)
        result["artifacts"]["date_mismatches_head"] = mismatches
        result["summary"]["n_date_mismatches"] = int(mismatch_mask.sum())
        result["ok"] = False
    else:
        result["summary"]["n_date_mismatches"] = 0

    # ------------------------------------------------------------
    # 3) Diferencias temporales globales
    # ------------------------------------------------------------
    # Trabajamos en segundos
    diffs_sec = pd.Series(df.index).diff().dt.total_seconds()

    n_non_positive_diffs = int((diffs_sec.iloc[1:] <= 0).sum())
    result["checks"]["all_global_time_diffs_positive"] = (n_non_positive_diffs == 0)
    result["summary"]["n_non_positive_global_diffs"] = n_non_positive_diffs

    if n_non_positive_diffs > 0:
        bad_diff_rows = df.iloc[np.where((diffs_sec <= 0).fillna(False))[0][:20]]
        result["artifacts"]["non_positive_global_diffs_head"] = bad_diff_rows
        result["ok"] = False

    # ------------------------------------------------------------
    # 4) Validación por día
    # ------------------------------------------------------------
    daily_stats = []
    bad_minute_order_days = []
    duplicate_minute_days = []
    intraday_gap_rows = []

    grouped = df.groupby("date", sort=False)

    for day, g in grouped:
        g = g.copy()

        # 4.1 orden del índice dentro del día
        idx_mono = g.index.is_monotonic_increasing

        # 4.2 minute_of_day creciente dentro del día
        mod_diff = g["minute_of_day"].diff()
        minute_order_ok = bool((mod_diff.iloc[1:] > 0).all())

        # 4.3 duplicados de minute_of_day dentro del día
        dup_mod = g["minute_of_day"].duplicated().sum()
        has_dup_mod = dup_mod > 0

        # 4.4 gaps intradía del índice
        idx_diff_sec = pd.Series(g.index).diff().dt.total_seconds()
        gap_mask = (~idx_diff_sec.isna()) & (idx_diff_sec != 60)

        n_intraday_gaps = int(gap_mask.sum())

        if not minute_order_ok:
            bad_minute_order_days.append(day)

        if has_dup_mod:
            duplicate_minute_days.append(day)

        if n_intraday_gaps > 0:
            gap_info = g.loc[gap_mask, ["date", "minute_of_day", "open", "high", "low", "close", "volume"]].copy()
            gap_info["gap_seconds"] = idx_diff_sec[gap_mask].values
            intraday_gap_rows.append(gap_info)

        daily_stats.append({
            "date": day,
            "n_rows": len(g),
            "index_monotonic": bool(idx_mono),
            "minute_of_day_monotonic": minute_order_ok,
            "n_duplicate_minute_of_day": int(dup_mod),
            "n_intraday_gaps_not_60s": n_intraday_gaps,
            "minute_min": int(g["minute_of_day"].min()),
            "minute_max": int(g["minute_of_day"].max()),
        })

        if not idx_mono or not minute_order_ok or has_dup_mod:
            result["ok"] = False

    daily_stats_df = pd.DataFrame(daily_stats)

    result["artifacts"]["daily_stats"] = daily_stats_df
    result["summary"]["n_days"] = int(daily_stats_df.shape[0])
    result["summary"]["days_with_bad_minute_order"] = int(len(bad_minute_order_days))
    result["summary"]["days_with_duplicate_minute_of_day"] = int(len(duplicate_minute_days))
    result["summary"]["days_with_intraday_gaps_not_60s"] = int((daily_stats_df["n_intraday_gaps_not_60s"] > 0).sum())

    result["checks"]["all_days_have_monotonic_minute_of_day"] = (len(bad_minute_order_days) == 0)
    result["checks"]["no_duplicate_minute_of_day_within_day"] = (len(duplicate_minute_days) == 0)
    result["checks"]["all_intraday_steps_are_60s_within_day"] = bool(
        (daily_stats_df["n_intraday_gaps_not_60s"] == 0).all()
    )

    result["artifacts"]["bad_minute_order_days"] = bad_minute_order_days
    result["artifacts"]["duplicate_minute_days"] = duplicate_minute_days

    if intraday_gap_rows:
        result["artifacts"]["intraday_gaps_head"] = pd.concat(intraday_gap_rows, axis=0).head(50)
    else:
        result["artifacts"]["intraday_gaps_head"] = pd.DataFrame()

    # ------------------------------------------------------------
    # 5) Resumen global
    # ------------------------------------------------------------
    result["summary"]["n_rows"] = int(len(df))
    result["summary"]["start"] = df.index.min()
    result["summary"]["end"] = df.index.max()

    # ------------------------------------------------------------
    # 6) Reporte por pantalla
    # ------------------------------------------------------------
    if verbose:
        print("=" * 70)
        print("VALIDACIÓN TEMPORAL DE mnq_intraday")
        print("=" * 70)
        print(f"Rows                     : {result['summary']['n_rows']}")
        print(f"Days                     : {result['summary']['n_days']}")
        print(f"Start                    : {result['summary']['start']}")
        print(f"End                      : {result['summary']['end']}")
        print("-" * 70)
        print(f"Index monotonic          : {result['checks']['index_is_monotonic_increasing']}")
        print(f"Index unique             : {result['checks']['index_is_unique']}")
        print(f"Date == index.date       : {result['checks']['date_column_matches_index_date']}")
        print(f"Global diffs > 0         : {result['checks']['all_global_time_diffs_positive']}")
        print(f"minute_of_day monotonic  : {result['checks']['all_days_have_monotonic_minute_of_day']}")
        print(f"No dup minute_of_day     : {result['checks']['no_duplicate_minute_of_day_within_day']}")
        print(f"Intraday steps = 60s     : {result['checks']['all_intraday_steps_are_60s_within_day']}")
        print("-" * 70)
        print(f"Duplicated timestamps    : {result['summary']['n_duplicated_timestamps']}")
        print(f"Date mismatches          : {result['summary']['n_date_mismatches']}")
        print(f"Non-positive global diffs: {result['summary']['n_non_positive_global_diffs']}")
        print(f"Bad minute order days    : {result['summary']['days_with_bad_minute_order']}")
        print(f"Dup minute_of_day days   : {result['summary']['days_with_duplicate_minute_of_day']}")
        print(f"Days with !=60s gaps     : {result['summary']['days_with_intraday_gaps_not_60s']}")
        print("-" * 70)
        print(f"DATASET OK               : {result['ok']}")
        print("=" * 70)

        if result["summary"]["n_duplicated_timestamps"] > 0:
            print("\nDuplicated timestamps (head):")
            print(pd.Index(result["artifacts"]["duplicated_timestamps"][:10]))

        if result["summary"]["n_date_mismatches"] > 0:
            print("\nDate mismatches (head):")
            print(result["artifacts"]["date_mismatches_head"])

        if result["summary"]["days_with_bad_minute_order"] > 0:
            print("\nDays with bad minute_of_day order (head):")
            print(result["artifacts"]["bad_minute_order_days"][:10])

        if result["summary"]["days_with_duplicate_minute_of_day"] > 0:
            print("\nDays with duplicate minute_of_day (head):")
            print(result["artifacts"]["duplicate_minute_days"][:10])

        if not result["artifacts"]["intraday_gaps_head"].empty:
            print("\nIntraday gaps != 60 seconds (head):")
            print(result["artifacts"]["intraday_gaps_head"])

    return result


# ============================================================
# Ejecución inmediata
# ============================================================
validation = validate_mnq_intraday(mnq_intraday, verbose=True)

# Si quiere abortar automáticamente cuando haya problemas críticos:
critical_checks = [
    "index_is_monotonic_increasing",
    "index_is_unique",
    "date_column_matches_index_date",
    "all_global_time_diffs_positive",
    "all_days_have_monotonic_minute_of_day",
    "no_duplicate_minute_of_day_within_day",
]

failed_critical = [k for k in critical_checks if not validation["checks"].get(k, False)]

if failed_critical:
    raise ValueError(
        "Validación temporal fallida. Checks críticos con error: "
        + ", ".join(failed_critical)
    )

VALIDACIÓN TEMPORAL DE mnq_intraday
Rows                     : 894845
Days                     : 1295
Start                    : 2020-01-02 04:30:00-05:00
End                      : 2025-06-13 16:00:00-04:00
----------------------------------------------------------------------
Index monotonic          : True
Index unique             : True
Date == index.date       : True
Global diffs > 0         : True
minute_of_day monotonic  : True
No dup minute_of_day     : True
Intraday steps = 60s     : True
----------------------------------------------------------------------
Duplicated timestamps    : 0
Date mismatches          : 0
Non-positive global diffs: 0
Bad minute order days    : 0
Dup minute_of_day days   : 0
Days with !=60s gaps     : 0
----------------------------------------------------------------------
DATASET OK               : True


# **2. Construcción de targets**

### **Construcción de targets**



In [15]:
import numpy as np
import pandas as pd


def compute_intraday_targets(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    date_col: str = "date",
    check_input: bool = True,
    overwrite: bool = False,
) -> pd.DataFrame:
    """
    Construye targets intradía sin cruzar días.

    Solo calcula:
      delta_60
      delta_90
    """

    horizons = (60, 90)

    if check_input:
        if not isinstance(df, pd.DataFrame):
            raise TypeError("df debe ser un pandas.DataFrame")

        if df.empty:
            raise ValueError("df está vacío")

        required_cols = [close_col, date_col]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Faltan columnas requeridas: {missing}")

        if not isinstance(df.index, pd.DatetimeIndex):
            raise TypeError("El índice debe ser un pd.DatetimeIndex")

        if not df.index.is_monotonic_increasing:
            raise ValueError("El índice datetime no está ordenado crecientemente")

        if not df.index.is_unique:
            raise ValueError("El índice datetime contiene duplicados")

        if df[close_col].isna().any():
            raise ValueError(f"La columna '{close_col}' contiene NaNs")

    out = df.copy()

    # Columnas a crear (solo delta)
    new_cols = [f"delta_{h}" for h in horizons]

    existing = [c for c in new_cols if c in out.columns]
    if existing and not overwrite:
        raise ValueError(
            f"Las siguientes columnas ya existen y overwrite=False: {existing}"
        )

    grouped_close = out.groupby(date_col, group_keys=False)[close_col]

    for h in horizons:
        p_t = out[close_col]
        p_th = grouped_close.shift(-h)

        out[f"delta_{h}"] = p_th - p_t

    return out

### **Validación de targets**

In [13]:
def validate_intraday_targets(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    date_col: str = "date",
    rtol: float = 1e-10,
    atol: float = 1e-12,
    verbose: bool = True,
) -> dict:
    """
    Valida:
    1) Fórmula de delta_h
    2) No cruce de días
    3) NaNs esperados por jornada
    """

    horizons = (60, 90)

    required_base = [close_col, date_col]
    missing_base = [c for c in required_base if c not in df.columns]
    if missing_base:
        raise ValueError(f"Faltan columnas base requeridas: {missing_base}")

    for h in horizons:
        col = f"delta_{h}"
        if col not in df.columns:
            raise ValueError(f"Falta la columna requerida: '{col}'")

    summary_rows = []
    day_sizes = df.groupby(date_col).size()

    for h in horizons:
        p_t = df[close_col]
        p_th = df.groupby(date_col, group_keys=False)[close_col].shift(-h)

        expected_delta = p_th - p_t
        valid_mask = p_t.notna() & p_th.notna()

        # Fórmula
        delta_ok = np.allclose(
            df.loc[valid_mask, f"delta_{h}"].to_numpy(),
            expected_delta.loc[valid_mask].to_numpy(),
            rtol=rtol,
            atol=atol,
        )

        assert delta_ok, f"delta_{h} incorrecto"

        # No cruce de días
        date_t = df[date_col]
        date_th = df.groupby(date_col, group_keys=False)[date_col].shift(-h)

        mask = date_th.notna()
        assert (date_th[mask] == date_t[mask]).all(), f"Cruce de día en h={h}"

        # NaNs esperados
        nan_counts = df[f"delta_{h}"].isna().groupby(df[date_col]).sum()
        expected_nans = day_sizes.clip(upper=h)

        assert (nan_counts == expected_nans).all(), f"NaNs incorrectos en h={h}"

        summary_rows.append({
            "h": h,
            "n_valid_obs": int(valid_mask.sum()),
            "delta_ok": bool(delta_ok),
        })

    summary_df = pd.DataFrame(summary_rows)
    all_ok = summary_df.drop(columns=["h", "n_valid_obs"]).all().all()

    if verbose:
        print("=" * 70)
        print("VALIDACIÓN DE TARGETS")
        print("=" * 70)
        print(summary_df.to_string(index=False))
        print("-" * 70)
        print(f"ALL TARGETS OK: {bool(all_ok)}")
        print("=" * 70)

    return {
        "ok": bool(all_ok),
        "summary": summary_df,
    }

### **Aplicación**

In [18]:
mnq_targets = compute_intraday_targets(
    mnq_intraday,

)

validation = validate_intraday_targets(
    mnq_targets,

)

if not validation["ok"]:
    raise ValueError("Targets inválidos")

VALIDACIÓN DE TARGETS
 h  n_valid_obs  delta_ok
60       817145      True
90       778295      True
----------------------------------------------------------------------
ALL TARGETS OK: True


In [19]:
mnq_targets

,date,open,high,low,close,volume,minute_of_day,is_premarket,is_opening,is_regular,is_closing,is_overnight,is_mon,is_tue,is_wed,is_thu,is_fri,delta_60,delta_90
datetime,,,,,,,,,,,,,,,,,,,
2020-01-02 04:30:00-05:00,2020-01-02,8813.25,8813.25,8812.50,8813.25,13,270,0,0,0,0,1,0,0,0,1,0,6.00,4.75
2020-01-02 04:31:00-05:00,2020-01-02,8812.50,8812.50,8811.00,8812.50,121,271,0,0,0,0,1,0,0,0,1,0,6.25,5.25
2020-01-02 04:32:00-05:00,2020-01-02,8812.50,8813.25,8811.75,8811.75,53,272,0,0,0,0,1,0,0,0,1,0,7.25,4.75
2020-01-02 04:33:00-05:00,2020-01-02,8812.00,8812.25,8810.50,8810.50,37,273,0,0,0,0,1,0,0,0,1,0,7.75,7.00
2020-01-02 04:34:00-05:00,2020-01-02,8810.75,8812.25,8810.50,8812.00,36,274,0,0,0,0,1,0,0,0,1,0,6.75,6.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,956,0,0,0,1,0,0,0,0,0,1,NaN,NaN
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,957,0,0,0,1,0,0,0,0,0,1,NaN,NaN
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,958,0,0,0,1,0,0,0,0,0,1,NaN,NaN


# **2. Indicadores Técnicos**

## **2.1. Indicadores técnicos individuales**

In [20]:
import pandas as pd
from ta.momentum import ROCIndicator, StochasticOscillator
from ta.volatility import AverageTrueRange

def calculate_technical_indicators(df: pd.DataFrame, target: str = "close") -> pd.DataFrame:
    """
    Calcula indicadores técnicos intradía por jornada.

    Indicadores calculados:
    - ema_60
    - roc_30
    - roc_60
    - stoch_k_20
    - atr_norm_10

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame de entrada.
    target : str, default="close"
        Columna objetivo sobre la cual se calculan EMA/ROC/Stochastic.

    Retorna
    -------
    pd.DataFrame
        DataFrame con los indicadores agregados.
    """

    technical_indicators_columns = [
        "ema_60",
        "roc_30",
        "roc_60",
        "stoch_k_20",
        "atr_norm_10",
    ]

    required_cols = [target, "high", "low", "date"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    if df.empty:
        raise ValueError("El DataFrame está vacío")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice debe ser un pd.DatetimeIndex")

    if not df.index.is_monotonic_increasing:
        raise ValueError("El índice datetime no está ordenado crecientemente")

    def aplicar_por_dia(grupo: pd.DataFrame) -> pd.DataFrame:
        grupo = grupo.copy()

        grupo["ema_60"] = grupo[target] / grupo[target].ewm(span=60, adjust=False).mean() - 1
        grupo["roc_30"] = ROCIndicator(close=grupo[target], window=30).roc()
        grupo["roc_60"] = ROCIndicator(close=grupo[target], window=60).roc()

        stoch_20 = StochasticOscillator(
            high=grupo["high"],
            low=grupo["low"],
            close=grupo[target],
            window=20,
            smooth_window=3,
        )
        grupo["stoch_k_20"] = stoch_20.stoch()

        atr_10 = AverageTrueRange(
            high=grupo["high"],
            low=grupo["low"],
            close=grupo[target],
            window=10,
        )
        grupo["atr_norm_10"] = atr_10.average_true_range() / grupo[target]

        return grupo

    out = (
        df.groupby("date", group_keys=False)
        .apply(aplicar_por_dia)
        .sort_index()
    )

    return out

## **2.2. Cálculo de indicadores técnicos**

Calculamos los indicadores técnicos

In [22]:
mnq_intraday_features = calculate_technical_indicators(
    df=mnq_targets,
    target="close"
)

In [24]:
info_mnq_features = mnq_dataset_info(mnq_intraday_features, name="mnq_intraday_features", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_features)

Dataset: mnq_intraday_features
Shape: (894845, 24)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'delta_90', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 04:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 270, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 09:30:00+00:00  ->  2025-06-13 20:00:00+00:00


Filtramos todos los NaNs del dataset

In [27]:
# Eliminación de filas con NaN
mnq_intraday_features = mnq_intraday_features.dropna()

In [28]:
info_mnq_features = mnq_dataset_info(mnq_intraday_features, name="mnq_intraday_features", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_features)

Dataset: mnq_intraday_features
Shape: (700595, 24)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'delta_90', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


In [29]:
assert not mnq_intraday.isna().any().any(), \
    "El dataset contiene NaN"

# **4. Remover columnas base sin valor**

Removemos las columnas base que no aportan valor `open`, `high`, `low` y `volume`:

In [33]:
regime_cols = [
    "is_premarket",
    "is_opening",
    "is_regular",
    "is_closing",
    "is_overnight",
]

regime_map = {
    "is_overnight": 0,
    "is_premarket": 1,
    "is_opening": 2,
    "is_regular": 3,
    "is_closing": 4,
}

mnq_intraday_features["regime_id"] = (
    mnq_intraday_features[regime_cols]
    .idxmax(axis=1)
    .map(regime_map)
)

cols_to_drop = [
    "is_premarket",
    "is_opening",
    "is_regular",
    "is_closing",
    "is_overnight",
    "is_mon",
    "is_tue",
    "is_wed",
    "is_thu",
    "is_fri",
]

# eliminar solo las que existan (evita errores)
mnq_intraday_features = mnq_intraday_features.drop(
    columns=[c for c in cols_to_drop if c in mnq_intraday_features.columns]
)

mnq_intraday_features.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day',
       'delta_60', 'delta_90', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20',
       'atr_norm_10', 'regime_id'],
      dtype='object')

In [ ]:
info_mnq = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq)

Dataset: mnq_intraday
Shape: (700595, 29)
Columns: ['date', 'close', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90', 'atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60', 'roc60_x_atr20', 'roc60_x_atr14', 'roc20_minus_roc60', 'mom5_minus_mom10']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


Obtenemos la hora de inicio y hora final

In [35]:
cols_order = [
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "minute_of_day",
    "regime_id",
    "ema_60",
    "roc_30",
    "roc_60",
    "stoch_k_20",
    "atr_norm_10",
    "delta_60",
    "delta_90",
]

mnq_intraday_features = mnq_intraday_features[cols_order]

In [36]:
mnq_intraday_features

,date,open,high,low,close,volume,minute_of_day,regime_id,ema_60,roc_30,roc_60,stoch_k_20,atr_norm_10,delta_60,delta_90
datetime,,,,,,,,,,,,,,,
2020-01-02 05:30:00-05:00,2020-01-02,8819.50,8819.75,8818.75,8819.25,83,330,0,0.000410,0.039702,0.068079,90.476190,0.000115,-6.50,-10.50
2020-01-02 05:31:00-05:00,2020-01-02,8819.25,8819.75,8818.50,8818.75,55,331,0,0.000342,0.034030,0.070922,80.952381,0.000118,-6.25,-10.00
2020-01-02 05:32:00-05:00,2020-01-02,8818.50,8819.00,8818.25,8819.00,36,332,0,0.000358,0.017012,0.082277,85.714286,0.000115,-8.25,-10.25
2020-01-02 05:33:00-05:00,2020-01-02,8819.00,8819.00,8818.25,8818.25,28,333,0,0.000264,0.025522,0.087963,71.428571,0.000112,-8.00,-8.50
2020-01-02 05:34:00-05:00,2020-01-02,8818.25,8819.00,8818.00,8818.75,27,334,0,0.000310,0.031193,0.076600,80.952381,0.000112,-8.25,-8.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,21716.50,21722.75,21712.00,21719.50,1036,866,3,-0.001801,-0.309818,-0.357839,30.739300,0.000671,-90.00,-102.00
2025-06-13 14:27:00-04:00,2025-06-13,21719.00,21719.75,21695.75,21698.50,3542,867,3,-0.002675,-0.406206,-0.419917,4.564315,0.000716,-69.25,-74.75
2025-06-13 14:28:00-04:00,2025-06-13,21698.25,21700.25,21670.50,21679.25,5241,868,3,-0.003444,-0.488852,-0.493419,10.233918,0.000782,-52.25,-57.50


In [37]:
import pandas as pd
import numpy as np
from pathlib import Path
import json


def validate_and_summarize_dataset(
    df: pd.DataFrame,
    *,
    date_col: str = "date",
    datetime_index_required: bool = True,
    save_path: str | None = None,
    verbose: bool = True,
) -> dict:
    """
    Valida calidad del dataset intradía y genera un resumen.

    Chequeos:
    - DataFrame no vacío
    - Índice datetime
    - Orden temporal
    - Duplicados
    - NaNs
    - Tamaño por día

    Retorna un dict con métricas clave y opcionalmente lo guarda en JSON.
    """

    summary = {}

    # -------------------------
    # 1. Validaciones básicas
    # -------------------------
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df debe ser un pandas.DataFrame")

    if df.empty:
        raise ValueError("El dataset está vacío")

    summary["n_rows"] = int(len(df))
    summary["n_cols"] = int(df.shape[1])
    summary["columns"] = list(df.columns)

    # -------------------------
    # 2. Índice temporal
    # -------------------------
    if datetime_index_required:
        if not isinstance(df.index, pd.DatetimeIndex):
            raise TypeError("El índice debe ser DatetimeIndex")

        summary["is_sorted"] = bool(df.index.is_monotonic_increasing)
        summary["has_duplicates_index"] = bool(not df.index.is_unique)

        assert summary["is_sorted"], "El índice no está ordenado"
        assert not summary["has_duplicates_index"], "Hay duplicados en el índice"

    # -------------------------
    # 3. NaNs
    # -------------------------
    nan_counts = df.isna().sum()
    summary["nan_total"] = int(nan_counts.sum())
    summary["nan_by_column"] = nan_counts[nan_counts > 0].to_dict()

    assert summary["nan_total"] == 0, f"Hay NaNs en el dataset: {summary['nan_by_column']}"

    # -------------------------
    # 4. Consistencia por día
    # -------------------------
    if date_col in df.columns:
        day_sizes = df.groupby(date_col).size()

        summary["n_days"] = int(day_sizes.shape[0])
        summary["rows_per_day_mean"] = float(day_sizes.mean())
        summary["rows_per_day_min"] = int(day_sizes.min())
        summary["rows_per_day_max"] = int(day_sizes.max())

    # -------------------------
    # 5. Estadísticas básicas
    # -------------------------
    numeric_cols = df.select_dtypes(include=np.number).columns

    summary["numeric_stats"] = {
        col: {
            "mean": float(df[col].mean()),
            "std": float(df[col].std()),
            "min": float(df[col].min()),
            "max": float(df[col].max()),
        }
        for col in numeric_cols
    }

    # -------------------------
    # 6. Guardado
    # -------------------------
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)

        with open(save_path, "w") as f:
            json.dump(summary, f, indent=4)

    # -------------------------
    # 7. Print
    # -------------------------
    if verbose:
        print("=" * 70)
        print("DATASET VALIDATION SUMMARY")
        print("=" * 70)
        print(f"Rows: {summary['n_rows']} | Cols: {summary['n_cols']}")
        print(f"NaNs: {summary['nan_total']}")
        print(f"Sorted: {summary.get('is_sorted', 'N/A')}")
        print(f"Days: {summary.get('n_days', 'N/A')}")
        print("-" * 70)

    return summary

In [38]:
summary = validate_and_summarize_dataset(
    mnq_intraday_features,
    save_path="reports/dataset_summary.json"
)

DATASET VALIDATION SUMMARY
Rows: 700595 | Cols: 15
NaNs: 0
Sorted: True
Days: 1295
----------------------------------------------------------------------


# **8. Guardado de datasets**


In [40]:
base_cols = [
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "minute_of_day",
    "regime_id",
    "ema_60",
    "roc_30",
    "roc_60",
    "stoch_k_20",
    "atr_norm_10",
]

# Dataset delta_60
mnq_delta_60 = mnq_intraday_features[base_cols + ["delta_60"]].copy()

# Dataset delta_90
mnq_delta_90 = mnq_intraday_features[base_cols + ["delta_90"]].copy()

In [41]:
OUT_PARQUET_DELTA_60 = Path(os.environ.get("OUT_PARQUET_DELTA_60", "data/features/mnq_delta_60.parquet"))
OUT_PARQUET_DELTA_90 = Path(os.environ.get("OUT_PARQUET_DELTA_90", "data/features/mnq_delta_90.parquet"))


OUT_PARQUET_DELTA_60 = DRIVE_DIR / OUT_PARQUET_DELTA_60
OUT_PARQUET_DELTA_90 = DRIVE_DIR / OUT_PARQUET_DELTA_90


In [42]:
for path, df in [
    (OUT_PARQUET_DELTA_60, mnq_delta_60),
    (OUT_PARQUET_DELTA_90, mnq_delta_90),
]:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=True)
    print(f"[OK] Guardado: {path}")

[OK] Guardado: /content/drive/MyDrive/neural_profit/data/features/mnq_delta_60.parquet
[OK] Guardado: /content/drive/MyDrive/neural_profit/data/features/mnq_delta_90.parquet
